In [ ]:
import sys
from pathlib import Path

def find_gptcast_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'gptcast').is_dir():
            return p
        if (p / 'GPTCast' / 'gptcast').is_dir():
            return (p / 'GPTCast').resolve()
        p = p.parent
    return start.resolve()

ROOT = find_gptcast_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Using ROOT:', ROOT)

from gptcast.models import GPTCast, VAEGANVQ
from gptcast.models.components import GPT, GPTCastConfig
from gptcast.data import Era5LandSwvl1DataModule
from gptcast.utils.plotting import plot_era5land as plot_miarad
from gptcast.utils.converters import swvl1_norm_to_phys
import numpy as np
import random
from datetime import datetime
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from einops import rearrange

# Reproducibility note:
# - `top_k=None` uses multinomial sampling (stochastic)
# - `top_k=1` uses greedy decoding (deterministic)
try:
    from lightning.pytorch import seed_everything
    seed_everything(42, workers=True)
except Exception:
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


## Load pretrained model and a soil moisture (swvl1) sequence of data from the dataset


In [ ]:
# ROOT is set in the first cell (auto-detected)

# First-stage checkpoint (tokenizer/decoder)
# IMPORTANT: This must match the first-stage model used during second-stage training.
first_stage_ckpt = ROOT / 'logs/train/runs/2026-03-01_18-29-57/checkpoints/epoch_038.ckpt'

# Second-stage checkpoints (GPTCast)
# Note: this SWVL1 run is a *16x16* GPTCast model (see experiment name / training config).
# Here we compare two checkpoints from the same 16x16 run (A vs B), not 8x8 vs 16x16.
stage2_run_dir = 'logs/train/runs/2026-03-02_12-09-21'
gpt_a_ckpt = ROOT / f'{stage2_run_dir}/checkpoints/epoch_000.ckpt'
gpt_b_ckpt = ROOT / f'{stage2_run_dir}/checkpoints/last.ckpt'

assert first_stage_ckpt.exists(), f"Missing: {first_stage_ckpt}"
assert gpt_a_ckpt.exists(), f"Missing: {gpt_a_ckpt}"
assert gpt_b_ckpt.exists(), f"Missing: {gpt_b_ckpt}"

# Build the transformer with the same architecture as training
transformer = GPT(
    vocab_size=1024,
    block_size=2048,
    n_layer=GPTCastConfig.n_layer,
    n_head=GPTCastConfig.n_head,
    n_embd=GPTCastConfig.n_embd,
)

first_stage = VAEGANVQ.load_from_pretrained(str(first_stage_ckpt), device=device).to(device).eval()

gpt_a = GPTCast.load_from_checkpoint(
    str(gpt_a_ckpt),
    transformer=transformer,
    first_stage=first_stage,
    map_location=device,
).to(device).eval()

gpt_b = GPTCast.load_from_checkpoint(
    str(gpt_b_ckpt),
    transformer=transformer,
    first_stage=first_stage,
    map_location=device,
).to(device).eval()


In [ ]:
# ROOT is set in the first cell (auto-detected)

# The original GPTCast notebook samples a random test sequence.
# For SWVL1 this can be hard to interpret (and can be ocean-dominated). Here we use a
# reproducible example centred over East China, with a small "smart" jitter search to
# keep the crop mostly over land.

import xarray as xr
from datetime import timedelta
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch.nn.functional as F

# Example start time (daily at 11:00 in this dataset)
T0 = datetime(2020, 4, 6, 11, 0)
SEQ_LEN = 7

# Data location (yearly NetCDF)
BASE_DIR = ROOT / 'data/0.1/1/land_surface'
NC_NAME = 'volumetric_soil_water_layer_1.nc'

# Training-style preprocessing
CLIP = (0.0, 0.8)
RESIZE = 720
CROP = 256

# East China center (degrees)
ROI_NAME = 'East China'
ROI_CENTER = dict(lat=34.0, lon=116.0)

# Smart-crop around ROI center (on resized grid)
MAX_MASK_FRAC = 0.40
SMART_ATTEMPTS = 50
JITTER_PX = 80

# Local RNG so this ROI crop is reproducible even if other cells also use randomness
rng = random.Random(42)


def open_year_ds(year: int) -> xr.Dataset:
    p = BASE_DIR / str(year) / NC_NAME
    assert p.exists(), f"Missing: {p}"
    return xr.open_dataset(p)


def sel_time_swvl1(ds: xr.Dataset, dt: datetime):
    t = np.datetime64(dt)
    try:
        return ds['swvl1'].sel(time=t)
    except Exception:
        return ds['swvl1'].sel(time=t, method='nearest')


# Load frames (physical units, NaN over ocean)
frames = []
ds_y = open_year_ds(T0.year)
for j in range(SEQ_LEN):
    dt_j = T0 + timedelta(days=j)
    da = sel_time_swvl1(ds_y, dt_j)
    frames.append(da.values.astype(np.float32, copy=False))
frames = np.stack(frames, axis=0)  # (S, lat, lon)

lat_vals = ds_y['latitude'].values
lon_vals = ds_y['longitude'].values

mask_native = np.isnan(frames[0])

# Clip (physical)
cmin, cmax = float(CLIP[0]), float(CLIP[1])
frames = np.nan_to_num(frames, nan=cmin)
frames = np.clip(frames, cmin, cmax)

# Resize to square canvas (paper-style)
x = torch.from_numpy(frames).unsqueeze(1)  # (S,1,H,W)
x = F.interpolate(x, size=(RESIZE, RESIZE), mode='bilinear', align_corners=False)
frames_r = x.squeeze(1).numpy()

m = torch.from_numpy(mask_native.astype(np.float32))[None, None, ...]
m = F.interpolate(m, size=(RESIZE, RESIZE), mode='nearest')
mask_r = m[0, 0].numpy().astype(bool)

# Normalize to [-1,1]
nmin, nmax = -1.0, 1.0
frames_n = (frames_r - cmin) / (cmax - cmin + 1e-12)
frames_n = frames_n * (nmax - nmin) + nmin

# Map ROI center (lat/lon) to resized pixel indices
lat_c = float(ROI_CENTER['lat'])
lon_c = float(ROI_CENTER['lon'])

lat_idx = int(np.argmin(np.abs(lat_vals - lat_c)))
lon_idx = int(np.argmin(np.abs(lon_vals - lon_c)))

y_center = int(round(lat_idx / (len(lat_vals) - 1) * (RESIZE - 1)))
x_center = int(round(lon_idx / (len(lon_vals) - 1) * (RESIZE - 1)))


def clamp(v, lo, hi):
    return int(max(lo, min(hi, v)))


best = None
best_frac = 1.0

for k in range(SMART_ATTEMPTS):
    dy = rng.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    dx = rng.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    y0 = clamp(y_center - CROP // 2 + dy, 0, RESIZE - CROP)
    x0 = clamp(x_center - CROP // 2 + dx, 0, RESIZE - CROP)

    frac = float(mask_r[y0 : y0 + CROP, x0 : x0 + CROP].mean())
    if frac < best_frac:
        best_frac = frac
        best = (y0, x0)
    if frac <= MAX_MASK_FRAC:
        best = (y0, x0)
        best_frac = frac
        break

if best is None:
    raise RuntimeError('Failed to select a crop')

y0, x0 = best
print(f"ROI {ROI_NAME} center(lat={lat_c}, lon={lon_c}) -> resized center(y={y_center}, x={x_center})")
print(f"Selected crop top-left(y0={y0}, x0={x0}), mask_frac={best_frac:.3f}")

# Global context plot (first frame) + crop box
# Note: this is the resized field (same canvas used for patch extraction).
global0 = np.ma.array(frames_r[0], mask=mask_r)
cm = plt.get_cmap('viridis').copy()
cm.set_bad(color='lightgray')

fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=160, layout='constrained')
ax.imshow(global0, cmap=cm, vmin=cmin, vmax=cmax, interpolation='nearest')
ax.add_patch(Rectangle((x0, y0), CROP, CROP, linewidth=1.0, edgecolor='red', facecolor='none'))
ax.set_title(f"Global swvl1 (resized) {T0}  ROI: {ROI_NAME}")
ax.axis('off')
plt.show()

# Build an 'el' dict like the dataloader output (batch_size=1) for downstream cells.
patch = frames_n[:, y0 : y0 + CROP, x0 : x0 + CROP]  # (S,H,W) in [-1,1]
mask_patch = mask_r[y0 : y0 + CROP, x0 : x0 + CROP]

image_hws = np.transpose(patch, (1, 2, 0))  # (H,W,S)
el = {
    'image': torch.from_numpy(image_hws.astype(np.float32, copy=False))[None, ...],
    'mask': torch.from_numpy(mask_patch)[None, ...],
    'file_path_': [T0.strftime('SM_%Y%m%d%H%M')],
}


In [ ]:
dt = datetime.strptime(el['file_path_'][0], 'SM_%Y%m%d%H%M')
mask = el['mask'].cpu().numpy().squeeze()
data = rearrange(el['image'].cpu().numpy().squeeze(), 'h w s -> s h w')
input_data = np.ma.array(data, mask=np.broadcast_to(mask, data.shape))


# Plot input soil moisture sequence


In [ ]:
# Plot the input soil moisture sequence
# Note: `input_data` is defined in the previous cell that converts `el`.
assert 'input_data' in globals(), 'input_data is not defined. Run the cell above that builds input_data first.'
plot_miarad(swvl1_norm_to_phys(input_data), title=f"Input seq. {dt}")


## Forecast the next days (context = 7 days)

Note: GPTCast inference uses up to 7 context steps by default (trained with an 8-step temporal window = 7 past + 1 predicted).


In [ ]:
def forecast_swvl1(
    model: GPTCast,
    input_sequence: np.ma.MaskedArray,
    steps: int = 7,
    temperature: float = 1.0,
    top_k=1,
    verbosity: int = 1,
):
    # Forecast future swvl1 frames.
    # Input must be normalized to [-1, 1], with shape (s, h, w).
    # The GPTCast inference code supports a maximum input sequence length of 7 steps.
    # - top_k=1  : greedy decoding (deterministic)
    # - top_k=None: multinomial sampling (stochastic)
    assert input_sequence.ndim == 3
    assert steps >= 1

    if input_sequence.shape[0] > 7:
        input_sequence = input_sequence[-7:]
        print('Input sequence is longer than 7 steps, only the last 7 steps are used.')

    # separate mask and data
    if isinstance(input_sequence, np.ma.MaskedArray):
        x_m = input_sequence.mask
        x = input_sequence.data
    else:
        x = input_sequence
        x_m = np.zeros_like(input_sequence, dtype=bool)

    # create output mask
    input_mask_sum = x_m.sum(axis=0).astype(bool)
    output_mask = np.broadcast_to(input_mask_sum, (steps, *input_mask_sum.shape))

    x = torch.tensor(x, dtype=torch.float32).to(model.device)
    x = rearrange(x, 's h w -> h w s')

    with torch.no_grad():
        result = model.predict_sequence(
            x,
            steps=steps,
            future=True,
            window_size=None,
            temperature=temperature,
            top_k=top_k,
            verbosity=verbosity,
        )

    y = result['pred_sequence_nopad'].cpu().numpy().squeeze().clip(-1, 1)
    if output_mask.any() or isinstance(input_sequence, np.ma.MaskedArray):
        y = np.ma.masked_array(y, mask=output_mask)
    return y


# Note: multi-step forecasts can be slow (token-by-token generation).
# For a quick check, set FORECAST_STEPS=1; for your target task use 7 (d+1 ... d+7).
FORECAST_STEPS = 1
TOP_K = 1
TEMPERATURE = 1.0

forecast_a = forecast_swvl1(gpt_a, input_data, steps=FORECAST_STEPS, temperature=TEMPERATURE, top_k=TOP_K, verbosity=1)
forecast_b = forecast_swvl1(gpt_b, input_data, steps=FORECAST_STEPS, temperature=TEMPERATURE, top_k=TOP_K, verbosity=1)


### plot the forecasts


In [ ]:
plot_miarad(swvl1_norm_to_phys(forecast_a), title="GPTCast (ckpt A) forecast")
plot_miarad(swvl1_norm_to_phys(forecast_b), title="GPTCast (ckpt B) forecast")


In [ ]:
# --- Quantitative sanity check (paper-style logic) ---
# This evaluates a single reproducible East China ROI sample (land-heavy) with: 7-day context + N-day target.
# For paper-level reporting, aggregate these metrics over many samples / dates.

CONTEXT_STEPS = 7
EVAL_STEPS = 1  # set to 7 for a full week; keep small for runtime
EVAL_SEQ_LEN = CONTEXT_STEPS + EVAL_STEPS

# Greedy decoding for reproducibility. Set to None for stochastic sampling.
TOP_K = 1
TEMPERATURE = 1.0

# Build an eval sample over the *same ROI logic* as above, but with a longer sequence
# so we have both context and future targets.
assert 'BASE_DIR' in globals(), 'Run the ROI extraction cell above first (defines BASE_DIR, ROI_CENTER, etc.).'

import xarray as xr
from datetime import timedelta
import torch.nn.functional as F

T0_EVAL = T0  # reuse the same start date as the visualization

def open_year_ds(year: int) -> xr.Dataset:
    p = BASE_DIR / str(year) / NC_NAME
    assert p.exists(), f"Missing: {p}"
    return xr.open_dataset(p)

def sel_time_swvl1(ds: xr.Dataset, dt: datetime):
    t = np.datetime64(dt)
    try:
        return ds['swvl1'].sel(time=t)
    except Exception:
        return ds['swvl1'].sel(time=t, method='nearest')

frames = []
ds_y = open_year_ds(T0_EVAL.year)
for j in range(EVAL_SEQ_LEN):
    dt_j = T0_EVAL + timedelta(days=j)
    da = sel_time_swvl1(ds_y, dt_j)
    frames.append(da.values.astype(np.float32, copy=False))
frames = np.stack(frames, axis=0)  # (S, lat, lon)

lat_vals = ds_y['latitude'].values
lon_vals = ds_y['longitude'].values
mask_native = np.isnan(frames[0])

# Clip + resize (same as training)
cmin, cmax = float(CLIP[0]), float(CLIP[1])
frames = np.nan_to_num(frames, nan=cmin)
frames = np.clip(frames, cmin, cmax)

x = torch.from_numpy(frames).unsqueeze(1)  # (S,1,H,W)
x = F.interpolate(x, size=(RESIZE, RESIZE), mode='bilinear', align_corners=False)
frames_r = x.squeeze(1).numpy()

m = torch.from_numpy(mask_native.astype(np.float32))[None, None, ...]
m = F.interpolate(m, size=(RESIZE, RESIZE), mode='nearest')
mask_r = m[0, 0].numpy().astype(bool)

# Normalize to [-1,1]
frames_n = (frames_r - cmin) / (cmax - cmin + 1e-12)
frames_n = frames_n * 2 - 1

# Map ROI center (lat/lon) to resized pixel indices
lat_c = float(ROI_CENTER['lat'])
lon_c = float(ROI_CENTER['lon'])
lat_idx = int(np.argmin(np.abs(lat_vals - lat_c)))
lon_idx = int(np.argmin(np.abs(lon_vals - lon_c)))
y_center = int(round(lat_idx / (len(lat_vals) - 1) * (RESIZE - 1)))
x_center = int(round(lon_idx / (len(lon_vals) - 1) * (RESIZE - 1)))

def clamp(v, lo, hi):
    return int(max(lo, min(hi, v)))

best = None
best_frac = 1.0
# Use the same local RNG seed as the visualization cell so crop is reproducible
rng_eval = random.Random(42)
for k in range(SMART_ATTEMPTS):
    dy = rng_eval.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    dx = rng_eval.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    y0 = clamp(y_center - CROP // 2 + dy, 0, RESIZE - CROP)
    x0 = clamp(x_center - CROP // 2 + dx, 0, RESIZE - CROP)
    frac = float(mask_r[y0 : y0 + CROP, x0 : x0 + CROP].mean())
    if frac < best_frac:
        best_frac = frac
        best = (y0, x0)
    if frac <= MAX_MASK_FRAC:
        best = (y0, x0)
        best_frac = frac
        break

y0, x0 = best
print(f"Eval ROI crop mask_frac={best_frac:.3f}  start={T0_EVAL}  context={CONTEXT_STEPS}  steps={EVAL_STEPS}")

patch = frames_n[:, y0 : y0 + CROP, x0 : x0 + CROP]  # (S,H,W)
mask_patch = mask_r[y0 : y0 + CROP, x0 : x0 + CROP]

seq_eval = np.ma.array(patch, mask=np.broadcast_to(mask_patch, patch.shape))
dt_eval = T0_EVAL
context = seq_eval[:CONTEXT_STEPS]
target = seq_eval[CONTEXT_STEPS:CONTEXT_STEPS + EVAL_STEPS]

def per_lead_mae_rmse(pred: np.ma.MaskedArray, tgt: np.ma.MaskedArray):
    diff = pred - tgt
    mae = np.ma.mean(np.abs(diff), axis=(1, 2))
    rmse = np.sqrt(np.ma.mean(diff ** 2, axis=(1, 2)))
    return mae, rmse

# Persistence baseline: repeat last context frame
persist = np.ma.array(
    np.broadcast_to(context[-1].data, target.shape),
    mask=target.mask,
)

pred_a = forecast_swvl1(gpt_a, context, steps=EVAL_STEPS, temperature=TEMPERATURE, top_k=TOP_K, verbosity=1)
pred_b = forecast_swvl1(gpt_b, context, steps=EVAL_STEPS, temperature=TEMPERATURE, top_k=TOP_K, verbosity=1)

# Compute metrics in physical units (m3/m3)
tgt_phys = swvl1_norm_to_phys(target)
pers_phys = swvl1_norm_to_phys(persist)
a_phys = swvl1_norm_to_phys(pred_a)
b_phys = swvl1_norm_to_phys(pred_b)

mae_p, rmse_p = per_lead_mae_rmse(pers_phys, tgt_phys)
mae_a, rmse_a = per_lead_mae_rmse(a_phys, tgt_phys)
mae_b, rmse_b = per_lead_mae_rmse(b_phys, tgt_phys)

print(f"Eval sample start: {dt_eval}  |  context={CONTEXT_STEPS}  steps={EVAL_STEPS}  top_k={TOP_K}")
print("lead  |  persistence_rmse  ckpt_A_rmse  ckpt_B_rmse")
for i in range(EVAL_STEPS):
    print(f"+{i+1:02d}   |  {rmse_p[i]:.4f}            {rmse_a[i]:.4f}       {rmse_b[i]:.4f}")

# Optional visualization: ground-truth vs predictions
plot_miarad(tgt_phys, title="Ground-truth future")
plot_miarad(a_phys, title="Predicted future (ckpt A)")
plot_miarad(b_phys, title="Predicted future (ckpt B)")
